# Notebook 08 — Environmental Anomaly Detection

**Project:** RiskWeave — climate-informed dengue risk forecasting and inspection planning for Sri Lanka.

## Purpose
Identify unusual weekly environmental conditions within each district using historical weather variables in the processed dengue-weather dataset.

Environmental anomaly scores use weather variables only. Dengue cases are not used to calculate these scores.

## Method
- District-specific trailing historical baselines.
- Exclude the current period from its baseline (`shift(1)`) to prevent look-ahead leakage.
- Rolling median and median absolute deviation (MAD).
- Separate robust scores for each weather variable.
- Screen both unusually high and unusually low values.
- The score threshold is provisional, not an official environmental or dengue outbreak threshold.

**Data-source note:** The historical weather data currently used in RiskWeave is Open-Meteo ERA5, not NASA POWER. Document this as a data-source deviation from the proposal.

An environmental anomaly is an unusual weather observation relative to that district's history. It does not establish increased dengue transmission, predict an outbreak, or confirm an outbreak.

## 1. Imports and settings

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROLLING_WINDOW = 52
MIN_BASELINE_OBSERVATIONS = 26
PROVISIONAL_ABS_SCORE_THRESHOLD = 3.5

print("Libraries loaded.")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 2. Resolve project paths and load the processed dataset

In [ ]:
cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError(
        "Could not identify RiskWeave project root. "
        "Open this notebook from the project notebooks folder or project root."
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dengue_weather_weekly_25_districts.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "environmental_anomaly_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
df["week_end"] = pd.to_datetime(df["week_end"], errors="coerce")

print("Dataset loaded.")
print("Project root:", PROJECT_ROOT)
print("Input:", DATA_PATH)
print("Output:", OUTPUT_DIR)
print("Shape:", df.shape)
print("Districts:", df["district_key"].nunique())
print("Date range:", df["week_start"].min(), "to", df["week_start"].max())
display(df.head())

## 3. Data-quality checks

In [ ]:
required_columns = ["district_key", "week_start", "week_end"]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

quality_report = {
    "rows": len(df),
    "districts": df["district_key"].nunique(),
    "missing_district": df["district_key"].isna().sum(),
    "missing_week_start": df["week_start"].isna().sum(),
    "missing_week_end": df["week_end"].isna().sum(),
    "duplicate_district_week_start": df.duplicated(["district_key", "week_start"]).sum(),
    "week_end_before_week_start": (df["week_end"] < df["week_start"]).sum(),
}
display(pd.Series(quality_report, name="value"))

if any(quality_report[k] > 0 for k in [
    "missing_district", "missing_week_start", "missing_week_end",
    "duplicate_district_week_start", "week_end_before_week_start"
]):
    raise ValueError("Resolve data-quality issues before continuing.")

print("Identifier and date checks passed.")

## 4. Select available environmental variables

Only columns present in the dataset will be used. Review any missing expected columns rather than assuming they exist.

In [ ]:
candidate_weather_features = [
    "precipitation_mean_daily",
    "precipitation_hours",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "cloud_cover_mean",
    "shortwave_radiation_mean_daily",
    "et0_fao_evapotranspiration",
]

weather_features = [c for c in candidate_weather_features if c in df.columns]
missing_weather_features = [c for c in candidate_weather_features if c not in df.columns]

print("Available weather features:")
for feature in weather_features:
    print("-", feature)

print("\nExpected weather features not found:")
for feature in missing_weather_features:
    print("-", feature)

if not weather_features:
    raise ValueError(
        "None of the expected weather columns were found. "
        "Inspect df.columns and update candidate_weather_features."
    )

print(f"Using {len(weather_features)} weather features.")

## 5. Inspect weather-variable completeness and distributions

Missing or non-numeric weather values are not silently imputed.

In [ ]:
for feature in weather_features:
    df[feature] = pd.to_numeric(df[feature], errors="coerce")

weather_quality = pd.DataFrame({
    "feature": weather_features,
    "missing_count": [int(df[c].isna().sum()) for c in weather_features],
    "missing_percent": [float(df[c].isna().mean() * 100) for c in weather_features],
    "minimum": [df[c].min() for c in weather_features],
    "median": [df[c].median() for c in weather_features],
    "maximum": [df[c].max() for c in weather_features],
})
display(weather_quality.sort_values("missing_percent", ascending=False))

## 6. Sort records and create calendar fields

In [ ]:
df = df.sort_values(["district_key", "week_start"]).reset_index(drop=True)
df["year"] = df["week_start"].dt.year
df["week_of_year"] = df["week_start"].dt.isocalendar().week.astype("Int64")

print("Chronological sorting completed.")
display(df[["district_key", "week_start", "week_of_year"] + weather_features].head())

## 7. Calculate district-specific trailing robust baselines

For each district and weather variable:
- Baseline uses up to the previous 52 reporting periods.
- Current period is excluded using `shift(1)`.
- At least 26 previous non-missing values are required.
- MAD is the median absolute deviation from the rolling median.

A zero MAD is not replaced with an arbitrary constant. Scores for zero-MAD baselines remain unavailable.

In [ ]:
scored_parts = []

for feature in weather_features:
    temp = df[["district_key", "week_start", "week_end", feature]].copy()
    temp = temp.sort_values(["district_key", "week_start"])
    historical = temp.groupby("district_key", sort=False)[feature].shift(1)

    temp[f"{feature}_baseline_median"] = (
        historical.groupby(temp["district_key"], sort=False)
        .transform(lambda s: s.rolling(
            window=ROLLING_WINDOW,
            min_periods=MIN_BASELINE_OBSERVATIONS
        ).median())
    )

    absolute_deviation = (historical - temp[f"{feature}_baseline_median"])

    temp[f"{feature}_baseline_mad"] = (
        absolute_deviation.groupby(temp["district_key"], sort=False)
        .transform(lambda s: s.rolling(
            window=ROLLING_WINDOW,
            min_periods=MIN_BASELINE_OBSERVATIONS
        ).median())
    )

    denominator = 1.4826 * temp[f"{feature}_baseline_mad"]
    temp[f"{feature}_robust_score"] = np.where(
        denominator > 0,
        (temp[feature] - temp[f"{feature}_baseline_median"]) / denominator,
        np.nan
    )

    scored_parts.append(temp[
        [
            "district_key", "week_start", "week_end", feature,
            f"{feature}_baseline_median",
            f"{feature}_baseline_mad",
            f"{feature}_robust_score"
        ]
    ])

print("Robust baselines and scores calculated.")

## 8. Reshape scores into a long-format table

Each row represents one district, reporting period and weather feature. Different weather variables retain their own units and scores.

In [ ]:
long_parts = []

for feature, part in zip(weather_features, scored_parts):
    feature_part = part.rename(columns={
        feature: "observed_value",
        f"{feature}_baseline_median": "baseline_median",
        f"{feature}_baseline_mad": "baseline_mad",
        f"{feature}_robust_score": "robust_score",
    }).copy()
    feature_part["weather_feature"] = feature
    long_parts.append(feature_part)

environmental_scores = pd.concat(long_parts, ignore_index=True)

environmental_scores["anomaly_status"] = np.select(
    [
        environmental_scores["robust_score"].isna(),
        environmental_scores["robust_score"] >= PROVISIONAL_ABS_SCORE_THRESHOLD,
        environmental_scores["robust_score"] <= -PROVISIONAL_ABS_SCORE_THRESHOLD,
    ],
    [
        "insufficient_or_zero_mad_baseline",
        "candidate_high_environmental_anomaly",
        "candidate_low_environmental_anomaly",
    ],
    default="not_flagged"
)

print("Environmental score table shape:", environmental_scores.shape)
display(environmental_scores.head(10))

## 9. Review score coverage and candidate counts

The provisional absolute score threshold screens both unusually high and unusually low weather values. It is not an official warning threshold.

In [ ]:
status_counts = (
    environmental_scores["anomaly_status"]
    .value_counts(dropna=False)
    .rename_axis("anomaly_status")
    .reset_index(name="observations")
)
display(status_counts)

feature_status_counts = (
    environmental_scores.groupby(["weather_feature", "anomaly_status"])
    .size().unstack(fill_value=0)
)
display(feature_status_counts)

## 10. Inspect the strongest candidate environmental anomalies

Very large scores can occur when historical MAD is small. Inspect the observed value, baseline median and MAD together.

In [ ]:
candidate_environmental_anomalies = environmental_scores[
    environmental_scores["anomaly_status"].isin([
        "candidate_high_environmental_anomaly",
        "candidate_low_environmental_anomaly",
    ])
].copy()

candidate_environmental_anomalies["absolute_robust_score"] = (
    candidate_environmental_anomalies["robust_score"].abs()
)

top_candidates = candidate_environmental_anomalies.sort_values(
    "absolute_robust_score", ascending=False
).head(30)

display(top_candidates[[
    "district_key", "week_start", "weather_feature",
    "observed_value", "baseline_median", "baseline_mad",
    "robust_score", "anomaly_status"
]])

## 11. Visualize one environmental variable for one district

Change `PLOT_FEATURE` to any feature listed in `weather_features`, and `PLOT_DISTRICT` to a district key present in the dataset.

In [ ]:
PLOT_FEATURE = "precipitation_mean_daily"
PLOT_DISTRICT = "colombo"

if PLOT_FEATURE not in weather_features:
    raise ValueError(f"{PLOT_FEATURE} is not available. Choose from: {weather_features}")

plot_data = environmental_scores[
    (environmental_scores["weather_feature"] == PLOT_FEATURE)
    & (environmental_scores["district_key"] == PLOT_DISTRICT)
].sort_values("week_start").copy()

if plot_data.empty:
    raise ValueError(f"No observations found for district: {PLOT_DISTRICT}")

plot_candidates = plot_data[
    plot_data["anomaly_status"].isin([
        "candidate_high_environmental_anomaly",
        "candidate_low_environmental_anomaly",
    ])
]

plt.figure(figsize=(15, 5))
plt.plot(plot_data["week_start"], plot_data["observed_value"],
         label="Observed weather value", linewidth=1)
plt.plot(plot_data["week_start"], plot_data["baseline_median"],
         label="Previous 52-period rolling median", linewidth=1.5)
plt.scatter(plot_candidates["week_start"], plot_candidates["observed_value"],
            label="Candidate environmental anomaly", s=25)
plt.title(f"Environmental anomaly screening — {PLOT_DISTRICT.title()} — {PLOT_FEATURE}")
plt.xlabel("Reporting period start")
plt.ylabel(PLOT_FEATURE)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 12. Compare candidate counts by district and weather feature

In [ ]:
candidate_counts = (
    candidate_environmental_anomalies
    .groupby(["district_key", "weather_feature"])
    .size()
    .unstack(fill_value=0)
)
display(candidate_counts)

plt.figure(figsize=(14, 7))
plt.imshow(candidate_counts.values, aspect="auto")
plt.colorbar(label="Candidate observations")
plt.xticks(range(len(candidate_counts.columns)), candidate_counts.columns,
           rotation=75, ha="right")
plt.yticks(range(len(candidate_counts.index)), candidate_counts.index)
plt.title("Candidate environmental anomalies by district and feature")
plt.xlabel("Weather feature")
plt.ylabel("District")
plt.tight_layout()
plt.show()

## 13. Export results and reproducibility manifest

In [ ]:
scores_path = OUTPUT_DIR / "environmental_anomaly_scores_weekly.csv"
candidates_path = OUTPUT_DIR / "candidate_environmental_anomalies.csv"
manifest_path = OUTPUT_DIR / "environmental_anomaly_manifest.json"

environmental_scores.to_csv(scores_path, index=False)
candidate_environmental_anomalies.to_csv(candidates_path, index=False)

manifest = {
    "project": "RiskWeave",
    "notebook": "08_environmental_anomaly_detection.ipynb",
    "input_file": str(DATA_PATH),
    "weather_data_source": "Open-Meteo ERA5 (as currently used in the project)",
    "rows_in_source_dataset": int(len(df)),
    "district_count": int(df["district_key"].nunique()),
    "date_start": str(df["week_start"].min().date()),
    "date_end": str(df["week_start"].max().date()),
    "weather_features_used": weather_features,
    "rolling_window_reporting_periods": ROLLING_WINDOW,
    "minimum_baseline_observations": MIN_BASELINE_OBSERVATIONS,
    "provisional_absolute_robust_score_threshold": PROVISIONAL_ABS_SCORE_THRESHOLD,
    "baseline_excludes_current_observation": True,
    "score_formula": "(observed_value - baseline_median) / (1.4826 * baseline_MAD)",
    "zero_or_missing_mad_scores": "Not scored; retained as unavailable",
    "interpretation": "Candidate unusual environmental observations only; not a dengue outbreak prediction or confirmation."
}
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("Files saved:")
print(scores_path)
print(candidates_path)
print(manifest_path)

## 14. Limitations and next validation tasks

Before operational use, environmental anomaly outputs need further review:

1. **Baseline sensitivity:** Investigate small and zero MAD values. Do not replace them with an arbitrary constant.
2. **Seasonality:** Compare the trailing baseline with district-specific seasonally matched baselines using historical observations only.
3. **Threshold validation:** Evaluate alternative thresholds and resulting candidate counts. A threshold is not an official warning criterion.
4. **Weather-data quality:** Review variable definitions, units, aggregation methods, missingness and source coverage.
5. **Spatial interpretation:** Compare district-specific patterns; national aggregation may hide local differences.
6. **Epidemiological interpretation:** Environmental anomalies are not dengue outbreaks. Any relationship with dengue must be investigated separately and with appropriate time lags.
7. **Data-source documentation:** Record that the current historical weather source is Open-Meteo ERA5, while the proposal identifies NASA POWER.

### Completion checklist
- [ ] Processed dataset loaded and paths verified
- [ ] Data-quality checks passed
- [ ] Available weather features reviewed
- [ ] Weather missingness and distributions inspected
- [ ] District-specific trailing baselines calculated
- [ ] Current observations excluded from their own baselines
- [ ] Candidate high and low environmental anomalies inspected
- [ ] District-level plots reviewed
- [ ] CSV outputs and manifest saved
- [ ] Limitations documented

**Do not describe environmental anomaly flags as confirmed dengue outbreaks.**